<a href="https://colab.research.google.com/github/josemolina6/16S-DADA2/blob/main/Do_it_yourself_Data_analysis_with_DADA2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Do it yourself: Data analysis with DADA2

hd**Background**

Advances in sequencing technology mean that it is now feasible to characterise eDNA and diet content through DNA-based approaches by simultaneously sequencing short standardised DNA sequences (DNA barcodes) from a variety of environmental samples, a process known as DNA metabarcoding.

This example dataset is from a study by [Doble et al. 2019](https://doi.org/10.1002/edn3.43) on tropical fish communities from Lake Tanganyika. The study's authors collected water eDNA samples from sites along the shore of Lake Tanganyika in Tanzania. Samples were then filtered, extracted and made into individual libraries using a two-step PCR method to attach indexes and Illumina adapters.

The data was generated using the MiFish-U primers ([Miya et al. 2015)](https://doi.org/10.1098/rsos.150088) that  target a hypervariable region of the 12S rRNA gene and sequenced using a single Illumina MiSeq 300 cycle sequencing run (2x150bp paired end sequencing).

The subset of samples included in this practical were sampled in triplicate from sites along the shore of Lake Tanganyika in 2017. Also included is a filter control, an extraction control and 5 aquarium samples. The aquarium samples contained five cichlid species endemic to the lake.

In this practical we run commands and set parameters that are appropriate for this example data set. When analysing your own data you will likely need to change the parameters used. For a more detailed explanation of each of the steps we recommend you read the [publication](https://doi.org/10.1038/nmeth.3869) and [manual](https://www.bioconductor.org/packages/release/bioc/manuals/dada2/man/dada2.pdf) for dada2.

Written by Katy Maher and Helen Hipperson, [NERC Environmental Omics Facility](https://neof.org.uk/), [School of Biosciences, University of Sheffield](https://www.sheffield.ac.uk/biosciences), UK

This Colab work is licensed under a [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International License](https://creativecommons.org/licenses/by-nc-sa/4.0/).

# Set up

This practical includes all the steps and code needed to install R packages and carry out metabarcoding analysis from the raw sequencing data through to assigning taxonomy and making some diversity plots. You can run through the analysis by clicking on the 'play' arrow button on the left of the blocks of code below, which will run the commands. It is important to wait until each command has finished running before moving on to the next one.


**Before you start**

This practical should take approximately **2 hours** to run from start to finish, including time to both run the code and read through the explanatory text. We recommend that you complete the practical in one session (although do take coffee breaks and give your eyes a rest from the screen!) as downloaded data and R objects may not remain available the following day or if you close down the tab.


Firstly, you'll need to install and load the required packages and download the example data.

Install the dada2 package

*This step takes approximately 5 minutes to run. Wait until it has completed (a green tick will appear on the left hand side) before proceeding to the next command.*

In [ ]:
if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager")
BiocManager::install("dada2")

Load the required R packages.

In [ ]:
library(dada2)
library(Biostrings)
library(ShortRead)

Download and unzip the example data files.

*This step takes approximately 2 minutes to run. Wait until it has completed (a green tick will appear on the left hand side) before proceeding to the next command.*

In [ ]:
system("wget https://ftp.ebi.ac.uk/pub/training/2024/Exploring-environmental-DNA/12S_data.tar.gz")

In [ ]:
list.files()

In [ ]:
# unzipping the files
system("tar -xvzf 12S_data.tar.gz")
list.files()

You should now have an unzipped folder called '12S_data', and you can proceed to the next section.

# DADA2 - checking data quality

We are now nearly ready to input our data but first we need to set the main paths we will be using which contain the input files (the raw sequencing data files) and where we want to save the output files we generate (the 'Metabarcoding' directory).

In [ ]:
input.path <- "/content/12S_data/cutadapt"
output.path <- "/content/Metabarcoding"

We can now check what files are contained in our input directory.

Run 'list.files' below and you should see a list of paired fastq sequencing files listed. Forward and reverse fastq filenames have format: `SAMPLENAME_L001_R1_001.fastq` and `SAMPLENAME_L001_R2_001.fastq`.

In [ ]:
list.files(input.path)

Inputting the forward and reverse reads:

We will assign the input path and specify all files which end in `_L001_R1_001.fastq` as our forward reads (stored as the variable fnFs) and `_L001_R2_001.fastq` as our reverse reads (stored as the variable fnRs).

In [ ]:
fnFs <- sort(list.files(input.path, pattern = "_L001_R1_001.fastq", full.names = TRUE))
fnRs <- sort(list.files(input.path, pattern = "_L001_R2_001.fastq", full.names = TRUE))

DADA2 has an in-built quality control option, which plots a read length by quality figure.

To run first make an object of sample names and then run the `plotQualityProfile` function:

In [ ]:
# Extract sample names
get.sample.name <- function(fname) strsplit(basename(fname), "_")[[1]][2]
sample.names <- unname(sapply(fnFs, get.sample.name))

# check the quality for the first file
plotQualityProfile(fnFs[1:1])

To interpret this plot, the gray-scale heatmap shows the the frequency of each quality score along the forward read lengths. The green line is the median quality score and the orange lines are the quartiles.

The red line at the bottom of the plot represents the proportion of reads of that particular length.

The quality is very good for our forward reads. You can also see that the majority of the forward reads are ~130bp long as the primers sequences were removed with cutadapt.

Now check the quality of the reverse file in the same way:

In [ ]:
plotQualityProfile(fnRs[1:1])

The reverse reads also look like they are good quality.

To check the quality of the second and third fastq files we would type:

In [ ]:
plotQualityProfile(fnFs[2:3])

plotQualityProfile(fnRs[2:3])

# DADA2 - cleaning your data

We will now filter our data to remove any poor quality reads.

First set the path to a directory to store the filtered output files called 'filtered'.

In [ ]:
filtFs <- file.path(output.path, "filtered", basename(fnFs))
filtRs <- file.path(output.path, "filtered", basename(fnRs))

Now run `filterAndTrim`. This time we use the standard filtering parameters:

- `maxN=0` After truncation, sequences with more than 0 Ns will be discarded. (DADA2 requires sequences contain no Ns)
- `truncQ = 2` Truncate reads at the first instance of a quality score less than or equal to 2
- `rm.phix = TRUE` Discard reads that match against the phiX genome
- `maxEE=c(2, 2)` After truncation, reads with higher than 2 "expected errors" will be discarded
- `minLen = 60` Remove reads with length less than 60
- `multithread = TRUE` input files are filtered in parallel

*This step takes approximately 5 minutes to run. Wait until it has completed (a green tick will appear on the left hand side) before proceeding to the next command.*

In [ ]:
out <- filterAndTrim(fnFs, filtFs, fnRs, filtRs, maxN = 0, maxEE = c(2, 2),
                     truncQ = 2, minLen = 60, rm.phix = TRUE, compress = TRUE,
                     multithread = TRUE)
out

This table shows for each sample the number of input reads from the raw files and the number remaining after the filtering.

Some samples have very low read numbers after this filtering step. These could be poor quality samples but we also have negatives controls in this dataset so we would expect these to contain zero or very few reads.

# DADA2 - identification of ASVs

**1. Generate an error model**

First we need to model the error rates of our dataset using both the forward and reverse reads. Each dataset will have a specific error-signature with errors introduced by PCR amplification and sequencing.

*This step takes approximately 15 minutes to run. Wait until it has completed (a green tick will appear on the left hand side) before proceeding to the next command.*

In [ ]:
errF <- learnErrors(filtFs, multithread = TRUE)
errR <- learnErrors(filtRs, multithread = TRUE)

We can use the `plotErrors` function to check the estimated error rates:

In [ ]:
plotErrors(errF, nominalQ = TRUE)


You will see some warning messages, but do not worry about them. This is a message from the plotting function to let you know that there were some zero values in the data plotted (which turn into infinities on the log-scale). This is expected, it results from the fact that not every combination of error type (e.g. A->C) and quality score (e.g. 33) is observed in your data, which is normal.

To interpret the plots:
- The error rates for each possible transition (e.g. A→C, A→G) are shown

- Red line - expected based on the quality score. (These are plotted when `nominalQ = TRUE` is included in the plot command)

- Black line - estimate

- Black dots - observed

What we are expecting to see here is that the observed match up with estimates. Here we can see that the black dots track well with the black line. We can also see that the error rates drop with increasing quality score as we would expect. Sanity checks complete, we can proceed with the analysis.

If you are worried about what your error plots look like when you fit your own dataset, one possible way to to improve the fit is to try increasing the number of bases the function is using (the default is 100 million).

**2. Dereplication**

The next step is to dereplicate identical reads. This is a common step in many workflows used for processing amplicons. This saves time and processing power as indentical reads are collapsed together. For example instead of processing 100 identical sequences, only one is processed but the original number (i.e. 100) is associated with it.

First we check that for samples that are still present after the filtering step and then perform the dereplication.

*This step takes approximately 2 minutes to run. Wait until it has completed (a green tick will appear on the left hand side) before proceeding to the next command.*

In [ ]:
exists <- file.exists(filtFs)
# check that all the samples are still present after filtering
derepFs <- derepFastq(filtFs[exists], verbose=TRUE)
derepRs <- derepFastq(filtRs[exists], verbose=TRUE)
# Name the derep-class objects by the sample names
names(derepFs) <- sample.names[exists]
names(derepRs) <- sample.names[exists]

**3. Inference of ASVs**

Now we are ready to infer the ASVs in our dataset. To do this DADA2 uses the error models created above to infer the true sample composition (follow [this link](https://www.nature.com/articles/nmeth.3869#methods) for more details).

Here we will run the inference algorithm on single samples to save time but it is also possible to run samples together in pseudo-pools to increase the ability to identify ASVs of low abundance. Low abundance ASVs in a sample may be filtered out when run separately but if they are found in higher numbers in another sample then the chance of that ASV being real increases. Pseudo-pooling increases the chances of these "real" low abundance ASVs being retained within samples. Whether or not to use the pseudo-pooling option will depend on your dataset and experimental design (see https://benjjneb.github.io/dada2/pseudo.html#Pseudo-pooling for more information).

It is important to note that you want to run the error and inference steps on datasets generated from a single Illumina run. Data generated from different runs can have different error structures.

*This step takes approximately 15 minutes to run. Wait until it has completed (a green tick will appear on the left hand side) before proceeding to the next command.*

In [ ]:
dadaFs <- dada(derepFs, err = errF, multithread = TRUE)
dadaRs <- dada(derepRs, err = errR, multithread = TRUE)

**4. Merging paired-end reads**

Up until now we have carried out all filtering, error correction and inference on the forward and reverse reads separately. It is now time to merge the two files. By default the minimum overlap allowed between the two samples is 12 bp.

In [ ]:
mergers <- mergePairs(dadaFs, derepFs, dadaRs, derepRs, verbose=TRUE)

**5. Making our ASV matrix**

Now it is time to make the counts table from the merged reads. Each column represents a single ASV and each row is an individual sample.



In [ ]:
seqtab <- makeSequenceTable(mergers)
dim(seqtab)

The `dim` function gives us the number of rows and columns in our matrix, so we can see that there are 3,218 ASVs in total.

**6. Chimera detection and removal**

The last step in generating our ASV matrix is to detect and remove any chimeric sequences.

Chimeric sequences are formed when two or more biological sequences are joined together. This is fairly common in amplicon sequencing. DADA2 uses a method whereby it combines the left and right segments of abundant reads and compares these with lower abundant sequences. Any low abundant sequences that match are removed.

In [ ]:
seqtab.nochim <- removeBimeraDenovo(seqtab, method="consensus",
                                    multithread=TRUE, verbose=TRUE)

dim(seqtab.nochim)

We can see that 2,857 ASVs have been identified as chimeric.

Although this is a large proportion of sequence variants it should be a smaller proportion of the total sequences.

Let's look at the proportion of sequences in the matrix with chimeras removed versus the original matrix:

In [ ]:
sum(seqtab.nochim)/sum(seqtab)

82% of our sequences remain, so in these data ~ 17% of the merged sequence reads were identified as chimeric.

We can also check the range of sequence lengths of our ASVs:

In [ ]:
table(nchar(getSequences(seqtab.nochim)))

Sequences that are much longer or shorter than the expected amplicon length could be the result of non-specific priming, and can be removed from your sequence table (e.g. `seqtab.len <- seqtab[,nchar(colnames(seqtab)) %in% 150:180]`). This is analogous to “cutting a band” in-silico to get amplicons of the targeted length, e.g. the above command would retain amplicons between 150 bp and 180 bp and remove others. This is dependent on the amplicon and you should only remove these if you do not expect any length variation, or after further investigation of what these short or long sequences are. For this example we will proceed without any amplicon length filtering.

**7. Sequence tracking sanity check**

The last thing to do in this section is to track the number of sequences through the pipeline to check whether everything has run as expected and whether there are any steps where we loose a disproportionate number of sequences. If we end up with too few reads to run further analysis we can use this table to identify any step which might require further investigation and optimisation.

In [ ]:
getN <- function(x) sum(getUniques(x))
track <- cbind(out, sapply(dadaFs, getN),
               sapply(dadaRs, getN),
               sapply(mergers, getN),
               rowSums(seqtab.nochim))
colnames(track) <- c("input", "filtered",
                     "denoisedF", "denoisedR",
                     "merged", "nonchim")
rownames(track) <- sample.names
track

This table shows us for each sample the number of sequences from the raw input files through the final ASV matrix.

# DADA2 - assigning taxonomy

To assign taxonomy we will use a custom reference database containing fish sequences available for the MiFish-U amplicon region for species found in Lake Tanganyika and its broader catchment area.

This reference database was included in the files you downloaded earlier. For more information about taxonomic assignments and database formatting see [this tutorial](https://benjjneb.github.io/dada2/assign.html).

In [ ]:
taxa <- assignTaxonomy(seqtab.nochim,
                       "/content/12S_data/MiFish_Reference_Database_taxonomy.fasta",
                       multithread=TRUE, verbose = T)

taxa.print <- taxa
rownames(taxa.print) <- NULL
head(taxa.print)

The `head` command displays the taxonomic assignment of the first six ASVs at taxonomic levels from kingdom to species. The second ASV has not been assigned to any taxon in the reference database. Of the other five, one has been assigned at the species level, three at the genus level and one at the family level.

# Further analysis


**1. Sample metadata and dataset clean up**

In this section we will briefly discuss a few basic types of multivariate data analysis and data visualisation which are often used in metabarcoding studies. It is important to remember that there are several different ways to plot and analyse your data and we have presented only a few examples here. It might be that for your own data set and questions it would be better to approach your analysis in a different way.

First we will load in a table which contains the metadata (was included in the files you downloaded earlier).

In [ ]:
meta<-read.csv("/content/12S_data/sample_info.csv",
               row.names = 1)

Before we start we will clean up the dataset. Our negative controls seem to be mostly clean of contaminants (with only 0 and 4 reads assigned to them). We will remove these for further analysis. Here we remove rows 25 and 31 which contain the filter (S62) and extraction (S72) control samples from our ASV table and metadata table.

In [ ]:
seqtab.rmcontrol<-seqtab.nochim[-c(25,31),]
meta.rmcontrol<-meta[-c(25,31),]

If you are concerned about contaminants in your own analysis then it could be useful to explore the R package [decontam](https://benjjneb.github.io/decontam/vignettes/decontam_intro.html) which is designed to identify and filter contaminants.

We will also remove samples which appear to have failed/have very low sequence numbers in our dataset. First lets check how many sequences are associated with each sample:


In [ ]:
#Check number of sequences per sample
rowSums(seqtab.rmcontrol)

The sample name is output along with the number of sequences.

S40, S41, S50 all have low sequence counts, so we will remove these rows.

In [ ]:
seqtab.rmlow<-seqtab.rmcontrol[-c(13,14,23),]
meta.rmlow<-meta.rmcontrol[-c(13,14,23),]

# Print the minimum sequence number in one sample.
min(rowSums(seqtab.rmlow))

The lowest number of sequences in a sample is now 26,664.

**2. R packages for further analysis**

We will use three more handy R packages in order to further explore our data, [phyloseq](https://joey711.github.io/phyloseq/) and [vegan](https://cran.r-project.org/web/packages/vegan/vegan.pdf) to explore the diverity in our data, plus the visualisation package ggplot2 for graphics. You will need to install and load these libraries first.

*This step takes approximately 15 minutes to run. Wait until it has completed (a green tick will appear on the left hand side) before proceeding to the next command.*

In [ ]:
install.packages(c('vegan', 'ggplot2'))

BiocManager::install("phyloseq")

In [ ]:
#Load the packages
library(phyloseq)
library(vegan)
library(ggplot2)

**3. Rarefaction curves**

The more deeply we sequence a sample the more species we discover, until this accumulation levels off as we have found all or as many ASVs/species as we are going to find. This becomes a problem as samples are often sequenced at different depths so will have reached different points in the curve. This is a common difficulty as pooling and sequencing equal amounts of each sample can be tricky.

One way to visualise this is to plot a rarefaction curve for each sample.

In [ ]:
rarecurve(seqtab.rmlow, step=100, col=meta.rmlow$COLOUR, lwd=2, ylab="ASVs", label=F)
# add a vertical line to represent the fewest sequences in any sample
abline(v=(min(rowSums(seqtab.rmlow))))

You can see that each sample levels off at a different sequencing depth and each sample has been sequenced at different depths. This also gives us an indication of how many ASVs are in each sample, with the aquarium samples here (dark blue lines) amongst the least diverse, as we'd expect.

Rarefaction is one of the common approaches people use to correct for differences in library sequencing depth between samples and was originally proposed for traditional ecological studies (Sanders 1968). It involves randomly removing reads until you reach a number often equal to or less than the number of reads in the smallest sample. Random subsampling can result in a loss of data and generate artificial variation. Due to these concerns other methods of transformation have been suggested.

Other methods of normalisation include normalising counts as a proportion of the total library size (McMurdie and Holmes 2014), centered log-ratio transformations (Gloor et al. 2017), geometric mean pairwise ratios (Chen et al. 2018), and also variance stabilising transformations and relative log expressions (Badri et al. 2018).

Choosing the type of normalisation method to use for your own dataset is not trivial and the most appropriate method can vary between studies and datasets. Further reading is recommended. A couple of papers to get you started are listed below:

-   [McMurdie and Holmes (2014)](https://journals.plos.org/ploscompbiol/article?id=10.1371/journal.pcbi.1003531)
-   [Willis (2019)](https://www.frontiersin.org/articles/10.3389/fmicb.2019.02407/full)
-   [Cameron *et al.* (2021)](https://www.nature.com/articles/s41598-021-01636-1)


**4. Alpha diversity**

Measures of alpha diversity are used to describe diversity within a sample.

We will use the R package [phyloseq](https://joey711.github.io/phyloseq/) to plot alpha diversity. For this example we will proceed with unnormalised data. It is generally recommended not to normalise count data before calculating alpha diversity measures in the [phyloseq FAQ](https://www.bioconductor.org/packages/devel/bioc/vignettes/phyloseq/inst/doc/phyloseq-FAQ.html#should-i-normalize-my-data-before-alpha-diversity-analysis).

First make a phyloseq object. To do this we first read in our ASV, taxonomy and metadata tables before making the plyloseq object `phylo`.

In [ ]:
seqtab.rmlow2<-t(as.data.frame(seqtab.rmlow))
phylo_asv <- otu_table(seqtab.rmlow2, taxa_are_rows=TRUE)
phylo_tax <- tax_table(taxa)
phylo_samples <- sample_data(meta.rmlow)

phylo <- phyloseq(phylo_asv, phylo_tax, phylo_samples)

sample_names(phylo)
rank_names(phylo)
sample_variables(phylo)

We will plot two alpha diversity metrics, Shannon's and Simpson's diversity index:

In [ ]:
plot_richness(phylo,
              measures=c("Shannon", "Simpson"),
              color = "SITE")
plot_richness(phylo, x="SITE", measures=c("Shannon", "Simpson"),
              color = "SITE") + geom_boxplot()

The first figure plots the diversity measure per sample and colours the output by site. The second figure combines the replicates to plot as a boxplot.

**5. Beta diversity**

Beta diversity compares the difference in diversity between two sites, or to put it another way it calculates the number of species that are not the same in the two sites.

We will normalise the data before running the beta diversity calculation. First we will transform the data into proportions to be used for Bray-Curtis distances:

In [ ]:
ps.prop <- transform_sample_counts(phylo, function(otu) otu/sum(otu))

We then generate and plot the NMDS (Non-metric MultiDimenstional Scaling) using Bray-Curtis distances.:

In [ ]:
ord.nmds.bray <- ordinate(ps.prop, method="NMDS", distance="bray")
plot_ordination(ps.prop, ord.nmds.bray, color="SITE", title="Bray NMDS")

In general the samples do seem to cluster roughly by site in the NMDS plot.

We will now calculate the Bray--Curtis distances using the `distance` function and perform a PERMANOVA (permutational multivariate analysis of variance) using the `adonis` function from Vegan to check whether the separation of samples by site is statistically significantly:

In [ ]:
bray.dist<-distance(ps.prop, method="bray")
sampledf <- data.frame(sample_data(phylo))
adonis2(bray.dist ~ SITE, data = sampledf)

The PERMANOVA results suggest that there is a statistical difference in communities between sites.

Lastly, let's plot the proportion of ASV sequences within each sample that belong to different taxonomic families:

In [ ]:
plot_bar(ps.prop, fill = "Family")+
  geom_bar(aes(color=Family, fill=Family), stat="identity", position="stack")+
  facet_grid(~SITE, scales = "free", space = "free")

We can see that the five aquarium samples contain mostly Cichlidae sequences, apart from a very small number of unassigned (NA) and misassigned (there were only cichlid fish in the aquarium) sequences.

Site 11 has more than 50% unassigned sequences in all three replicate samples.

All of the samples containing \> 10% Procatopodidae sequences are grouped together in the NMDS plot (especially sites 19 & 20, but also two samples from site 2, and the one sample from site 18), suggesting this may be in part driving the observed patterns of community similarity.

# Conclusion

Great work on reaching the end of the practical!

We hope that you have enjoyed getting some hands-on experience of metabarcoding data analysis.